# 🤖 LLM Scoring (Typhoon 2.5)

Scores scraped articles using local LLM + keyword detection:

**Keyword Columns (regex-based):**
| Column | Checks for |
|--------|------------|
| A.I. | AI, A.I., artificial intelligence |
| Google | Google |
| Microsoft | Microsoft |
| Nvidia | Nvidia |
| Crypto | Crypto, Bitcoin, Ethereum |
| Chipset | Chipset, chip, semiconductor |

**LLM-Scored Features:**
| Feature | Description |
|---------|-------------|
| Social_Impact | Does this affect society? |
| Economic_Impact | Does this affect markets/economy? |
| No_Ads | Is this real content (not advertisement)? |
| Has_Reference | Does it cite sources/data/research? |

**Output:** `scored_news.csv`

In [ ]:
# CELL 1: IMPORTS & CONFIG
import os
import json
import re
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm

# llama-cpp-python for local LLM
try:
    from llama_cpp import Llama
    print("✅ llama-cpp-python loaded")
except ImportError:
    print("❌ Install: pip install llama-cpp-python")
    raise

# CONFIG
class CFG:
    # Paths
    INPUT_CSV = "scraped_news_v10.csv"
    OUTPUT_CSV = "scored_news.csv"
    MODEL_PATH = "../data/models/typhoon2.5-qwen3-4b-q4_k_m.gguf"
    
    # TESTING: Only process first N articles (set to None for all)
    TEST_LIMIT = 15
    
    # Model settings
    N_CTX = 4096
    N_GPU_LAYERS = -1
    TEMPERATURE = 0.1
    MAX_TOKENS = 200
    
    # 6 KEYWORDS (same as scraper)
    KEYWORDS = ["A.I.", "Google", "Microsoft", "Nvidia", "Crypto", "Chipset"]
    
    # 4 LLM-scored features
    LLM_FEATURES = ["Social_Impact", "Economic_Impact", "No_Ads", "Has_Reference"]

print(f"📂 Input: {CFG.INPUT_CSV}")
print(f"📂 Output: {CFG.OUTPUT_CSV}")
print(f"🔑 Keywords: {CFG.KEYWORDS}")
print(f"🤖 LLM Features: {CFG.LLM_FEATURES}")
print(f"🧪 Test limit: {CFG.TEST_LIMIT} articles")

In [ ]:
# CELL 2: KEYWORD DETECTION (no LLM needed)
def detect_keywords(text):
    """
    Detect which keywords are present in the text.
    Returns dict with 0/1 for each keyword.
    """
    if not text:
        return {kw: 0 for kw in CFG.KEYWORDS}
    
    text_lower = text.lower()
    results = {}
    
    for kw in CFG.KEYWORDS:
        kw_lower = kw.lower()
        
        # Special case: "A.I." also matches "AI"
        if kw_lower == "a.i.":
            if "a.i." in text_lower or re.search(r'\bai\b', text_lower):
                results[kw] = 1
            else:
                results[kw] = 0
        else:
            results[kw] = 1 if kw_lower in text_lower else 0
    
    return results

# Test
test_text = "Google announces new AI chip for Microsoft Azure"
print(f"Test: '{test_text}'")
print(f"Keywords: {detect_keywords(test_text)}")

In [ ]:
# CELL 3: LOAD MODEL
print("Loading Typhoon 2.5 (may take 30-60 seconds)...")

llm = Llama(
    model_path=CFG.MODEL_PATH,
    n_ctx=CFG.N_CTX,
    n_gpu_layers=CFG.N_GPU_LAYERS,
    verbose=False
)

print("✅ Model loaded!")

In [ ]:
# CELL 4: LLM SCORING PROMPT (IMPROVED with clear criteria)
SCORING_PROMPT = """You are a news analyst. Analyze this article and score 4 features as 0 or 1.

ARTICLE:
Title: {title}
Content: {content}

SCORING CRITERIA:

1. Social_Impact:
   - Score 1 if: affects public policy, privacy, jobs, education, health, environment, or daily life of many people
   - Score 0 if: only affects businesses, investors, or niche technical audience

2. Economic_Impact:
   - Score 1 if mentions: funding round, IPO, stock price, revenue, layoffs, acquisition, valuation, market cap, earnings
   - Score 0 if: just product announcement, feature update, or technical news without financial implications

3. No_Ads:
   - Score 1 if: genuine news article with journalistic content
   - Score 0 if: sponsored content, press release, advertisement, promotional material, or "paid partnership"

4. Has_Reference:
   - Score 1 if: cites specific numbers, statistics, quotes from named people, research studies, or official sources
   - Score 0 if: vague claims, no specific data, no named sources

Respond ONLY with valid JSON:
{{"Social_Impact": 0, "Economic_Impact": 0, "No_Ads": 1, "Has_Reference": 0}}

JSON:"""

print("✅ Improved scoring prompt defined")

In [ ]:
# CELL 5: LLM SCORING FUNCTION
def score_with_llm(title: str, content: str) -> dict:
    """
    Score article on 4 features using LLM.
    """
    content_truncated = content[:2000] if content else ""
    prompt = SCORING_PROMPT.format(title=title, content=content_truncated)
    
    try:
        response = llm(
            prompt,
            max_tokens=CFG.MAX_TOKENS,
            temperature=CFG.TEMPERATURE,
            stop=["}", "\n\n"],
            echo=False
        )
        
        text = response['choices'][0]['text'].strip()
        if not text.endswith('}'):
            text = text + '}'
        
        scores = json.loads(text)
        
        result = {}
        for feature in CFG.LLM_FEATURES:
            val = scores.get(feature, 0)
            result[feature] = 1 if val == 1 or val == "1" or val == True else 0
        
        return result
        
    except json.JSONDecodeError:
        return {f: 0 for f in CFG.LLM_FEATURES}
    except Exception as e:
        print(f"Error scoring: {e}")
        return {f: 0 for f in CFG.LLM_FEATURES}

print("✅ LLM scoring function ready")

In [ ]:
# CELL 6: PROCESS ALL ARTICLES
df_full = pd.read_csv(CFG.INPUT_CSV)
print(f"📰 Loaded {len(df_full)} articles total")

if CFG.TEST_LIMIT:
    df = df_full.head(CFG.TEST_LIMIT)
    print(f"🧪 Testing with first {len(df)} articles")
else:
    df = df_full
    print(f"📊 Processing all {len(df)} articles")

results = []

print(f"\n🤖 Processing...")
print(f"⏱️ Estimated time: ~{len(df) * 2} seconds")
print()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    title = row.get('headline', '') or ''
    content = row.get('full_content', '') or row.get('content_snippet', '') or ''
    full_text = f"{title} {content}"
    
    # 1. Detect keywords (no LLM)
    keyword_scores = detect_keywords(full_text)
    
    # 2. LLM scoring (4 features)
    llm_scores = score_with_llm(title, content)
    
    # 3. Build result row
    result = {
        'news_id': idx,
        'url_hash': row.get('url_hash', ''),
        'headline': title[:100],
        **keyword_scores,
        **llm_scores
    }
    results.append(result)

print(f"\n✅ Processed {len(results)} articles")

In [ ]:
# CELL 7: SAVE RESULTS
scored_df = pd.DataFrame(results)

columns = ['news_id', 'url_hash', 'headline'] + CFG.KEYWORDS + CFG.LLM_FEATURES
scored_df = scored_df[columns]

scored_df.to_csv(CFG.OUTPUT_CSV, index=False)
print(f"💾 Saved to {CFG.OUTPUT_CSV}")

print(f"\n📊 Sample output:")
scored_df.head(10)

In [ ]:
# CELL 8: ANALYTICS
print("📊 Scoring Summary")
print("="*60)

print("\n🔑 KEYWORD COUNTS:")
for kw in CFG.KEYWORDS:
    count = scored_df[kw].sum()
    pct = (count / len(scored_df)) * 100
    print(f"   {kw:12} {count:4} articles ({pct:5.1f}%)")

print("\n🤖 LLM FEATURE COUNTS:")
for feature in CFG.LLM_FEATURES:
    count = scored_df[feature].sum()
    pct = (count / len(scored_df)) * 100
    print(f"   {feature:20} {count:4} articles ({pct:5.1f}%)")

print("\n" + "="*60)

# High-quality: No_Ads=1 AND Has_Reference=1 AND Economic_Impact=1
high_quality = scored_df[(scored_df['No_Ads'] == 1) & (scored_df['Has_Reference'] == 1)]
print(f"\n🏆 High-Quality News: {len(high_quality)} articles")
print("   (No_Ads=1 AND Has_Reference=1)")